# HLO from JAX
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week3/JAX_HLO_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 3, Module 4: XLA on TPU**  
**Author:** Dr. Kaoutar El Maghraoui

XLA compiles whole programs written in HLO, a small, high-level tensor IR. Its operations say what to compute, such as `dot`, `broadcast` and `maximum`, not how to loop. Fusion is decided at this level, before any loops exist, and the small operator set is what makes whole-program rewriting practical. JAX is the native path into XLA: `jax.jit` traces a Python function into a jaxpr, lowers it to StableHLO, and hands it to XLA.

In this notebook you print each stage for the function from the lecture, `relu(x @ w + b)`, then for a small attention computation. You count the fusions XLA chose, and you see the cost of XLA's commitment to static shapes: a new shape is a new trace and a new compile.

**What you will do**
1. Print the jaxpr, JAX's own graph format.
2. Print the StableHLO that XLA receives.
3. Print the HLO after XLA's optimization passes and find the fusions.
4. Compile an attention computation and count its fusions.
5. Call a jitted function with new shapes and count the traces.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Colab includes JAX with GPU support. On a CPU runtime the notebook still runs, but the optimized HLO looks different. The notebook takes about a minute.

## 1. Check the environment

In [ ]:
try:
    import jax
except ImportError:                      # Colab includes JAX; this installs it elsewhere
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jax"], check=True)
    import jax
import jax.numpy as jnp

print("JAX:", jax.__version__)
print("Devices:", jax.devices())
backend = jax.default_backend()
print("Backend:", backend)

## 2. The jaxpr
`jax.make_jaxpr` traces the function with abstract values and prints JAX's own graph. Every value has a dtype and shape, such as `f32[512,3072]`.

In [ ]:
def f(x, w, b):
    return jax.nn.relu(x @ w + b)


key = jax.random.PRNGKey(0)
kx, kw, kb = jax.random.split(key, 3)
x = jax.random.normal(kx, (512, 768), jnp.float32)
w = jax.random.normal(kw, (768, 3072), jnp.float32)
b = jax.random.normal(kb, (3072,), jnp.float32)

print(jax.make_jaxpr(f)(x, w, b))

**Reading the output**
- `dot_general` is the matrix multiply. Its `dimension_numbers` say which dimensions are contracted.
- The bias `b` has shape `[3072]`. Adding it to a `[512, 3072]` matrix needs a `broadcast_in_dim` first. In PyTorch this broadcast is implicit.
- `relu` appears as `max` with 0.0, sometimes inside a `pjit` or `custom_jvp_call` wrapper. There is no ReLU operator at this level.

## 3. StableHLO: what XLA receives
`jax.jit(f).lower(...)` produces StableHLO, the portable and versioned form of HLO. This is the module that XLA compiles.

In [ ]:
lowered = jax.jit(f).lower(x, w, b)
stablehlo = lowered.as_text()
for line in stablehlo.splitlines():
    if "stablehlo." in line:
        print(line.strip()[:120])

**Reading the output**
- `stablehlo.dot_general`, `stablehlo.broadcast_in_dim`, `stablehlo.add` and `stablehlo.maximum`. The lecture shows the same operations.
- There are no loops. Each operation declares a whole-tensor computation, and the compiler decides later how to loop over it.

## 4. Optimized HLO: what XLA did
`lowered.compile()` runs XLA's optimization passes and builds the executable. `as_text()` on the result prints the HLO after those passes. The cell lists the fusion computations and any calls out to libraries.

In [ ]:
compiled = lowered.compile()
hlo = compiled.as_text()


def summarize(hlo_text):
    fusions = [l.strip() for l in hlo_text.splitlines() if " fusion(" in l and "=" in l]
    calls = [l.strip() for l in hlo_text.splitlines() if "custom-call(" in l]
    return fusions, calls


fusions, calls = summarize(hlo)
print(f"{len(fusions)} fusion(s):")
for l in fusions:
    print("   ", l[:150])
print(f"{len(calls)} library call(s):")
for l in calls:
    print("   ", l[:150])

**Reading the output**
- On a GPU, the matrix multiply becomes a `custom-call` to cuBLASLt (`__cublas$lt$matmul`). The call takes three operands: `x`, `w` and the bias `b`.
- There are no fusions left. XLA folded the bias add and the ReLU into the matmul call itself, as an **epilogue** that runs before the result is written. The output crosses the memory bus once. This is the bottom row of the lecture's fusion slide, where the PyTorch three-IR demo in this folder reaches the middle row with Inductor's default settings.
- On a TPU, the same HLO would be mapped onto the matrix unit instead of a library call. The HLO module is the same, and the backend decides what each operation becomes.

## 5. An attention computation
A single attention head: scores `q @ k^T`, a scale, a softmax and a weighted sum with `v`. In eager PyTorch this is about seven kernels. The cell counts what XLA produces.

In [ ]:
def attention(q, k, v):
    scores = q @ k.T / jnp.sqrt(q.shape[-1])
    return jax.nn.softmax(scores, axis=-1) @ v


kq, kk, kv = jax.random.split(jax.random.PRNGKey(1), 3)
q = jax.random.normal(kq, (512, 64))
k = jax.random.normal(kk, (512, 64))
v = jax.random.normal(kv, (512, 64))

attn_hlo = jax.jit(attention).lower(q, k, v).compile().as_text()
fusions, calls = summarize(attn_hlo)
print(f"Attention: {len(fusions)} fusion(s), {len(calls)} library call(s)")
for l in fusions + calls:
    print("   ", l[:150])

**Reading the output**
- The softmax is a reduction (the row maximum), an exponential, another reduction (the row sum) and a division. On a GPU, XLA produces three fusions: one for the row maximum, one for the exponentials and their row sum, and one loop that computes the final division. The `kind=kInput` fusions are reductions and `kind=kLoop` is elementwise.
- The two matmuls are cuBLASLt calls. The scale does not appear as a separate kernel.
- This is XLA's design choice from the lecture: fuse aggressively across the whole program, at the level of tensor operations.

## 6. New shapes mean new traces
`jax.jit` caches one compiled program per combination of input shapes and dtypes. The `print` inside the function runs only while JAX traces it, so each printed line is a trace and a compile.

In [ ]:
@jax.jit
def g(x):
    print("  tracing for shape", x.shape)   # runs only during tracing
    return jnp.tanh(x) * 2.0


for n in (128, 128, 129, 130, 128):
    print(f"call with {n} columns")
    g(jnp.ones((8, n))).block_until_ready()

**Reading the output**
- The second call with 128 columns reuses the compiled program and prints nothing.
- 129 and 130 each trace and compile again. Unlike PyTorch's automatic dynamic shapes, `jax.jit` does not generalize a dimension after a mismatch. Every new shape is a new program.
- The last call with 128 reuses the first program from the cache.
- This is the trade from the lecture: XLA fits static, dense work, and pays more when shapes keep changing the cache key. Serving systems pad to fixed buckets for this reason.

## Try this
1. In section 4, print the whole optimized module with `print(hlo)` and find the layout annotations such as `{1,0}` on each tensor.
2. In section 5, use `jax.nn.gelu` instead of the softmax and count the fusions again.
3. Run the notebook on a CPU runtime and compare section 4: on the CPU there is no cuBLAS, so the `dot` stays in the HLO.

## Summary
- JAX traces Python into a jaxpr, lowers it to StableHLO, and XLA optimizes the HLO.
- HLO operations declare whole-tensor computations. Fusion is decided at that level, before loops exist.
- On a GPU, XLA calls cuBLAS for matmuls and fuses the elementwise and reduction work around them. On a TPU, the same HLO maps to the matrix unit.
- `jax.jit` specializes to input shapes. A new shape is a new trace and compile.

Lecture: Week 3, Module 4, slides 61 to 65 (why a TPU compiler, XLA by the numbers, HLO, two paths into XLA, when XLA wins).